## RAG: Foobar Mdg. Rules and Regulations Query

- 内部文書を読み込み、ChromaDBへembeddingsとcontentsを格納  
- 質問に関連のある内部文書を検索、検索結果を元に回答を試みる

pip install google-genai chromadb langchain-community  

### Vector Storeの作成

In [1]:
import os, sys
from pprint import pprint
from importlib.metadata import version

import google.genai as genai
import chromadb
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
print(f"{sys.version}")
print(f"google.genai={version('google-genai')}, chromadb={version('chromadb')}, ")
print(f"langchain_community={version('langchain-community')}, langchain_text_splitters={version('langchain-text-splitters')}, ")

# ====== Gemini の設定 ======
# Gemini API Keyを取得
with open('./GOOGLE_API_KEY.txt', 'r') as f:  # ファイルからAPI Keyを取得
    api_key = f.read().strip()
# Geminiモデルを指定
GEMINI_LLM_MODEL       = 'gemini-flash-latest'
GEMINI_EMBEDDING_MODEL = 'gemini-embedding-001'
# Geminiクライアントの作成
GEMINI_CLIENT = genai.Client(api_key=api_key)

# ====== ChromaDB の設定 ======
# clientの作成
# 　メモリ上だけ (永続化しない), 
# 　永続化させる場合には、chromadb.PersistentClient(path='save_to')を使用)
chroma_client = client = chromadb.EphemeralClient()

# ChromaDBの初期化、ローディング  ======================
# DB内collectionの初期化(既存のcollectionがあったら削除)
collection_name = 'novels'   # 任意の文字列
try:
    chroma_client.delete_collection(collection_name)
except:
    pass
collection = chroma_client.create_collection(
    name=collection_name,
    metadata={"hnsw:space": "cosine"}  # コサイン距離を指定
)

# 内部文書をロード ======================
target_dir = './Foobar_Mfg_Rules_and_Regulations/'
loader = DirectoryLoader(
    target_dir, 
    glob='*.md',
    loader_cls=TextLoader,
    loader_kwargs={'encoding': 'utf-8',},
)
documents = loader.load()   # ドキュメント(メタデータ付きテキスト)としてロード
print(f"Document files={len(documents)}")

# Chunks -> embedding/VectorStoreへ追加 処理用バッファ
pending_chunks = []   # 蓄積チャンク (Document オブジェクト)
pending_ids    = []   # 対応するID

# 先頭の BATCH_SIZE 件を embedding → collection.add() し、残りを返す関数
def embed_and_add_batch(chunks, ids):
    """"""
    batch_chunks = chunks[:BATCH_SIZE]
    batch_ids    = ids[:BATCH_SIZE]

    embedded = GEMINI_CLIENT.models.embed_content(
        model    = GEMINI_EMBEDDING_MODEL,
        contents = [c.page_content for c in batch_chunks]
    )
    chunk_vectors = [e.values for e in embedded.embeddings]
    collection.add(
        ids        = batch_ids,
        embeddings = chunk_vectors,
        documents  = [c.page_content for c in batch_chunks],
        metadatas  = [c.metadata    for c in batch_chunks]
    )
    print(f"  -> {len(batch_chunks)} chunks added (total={len(chunks)}, remaining={len(chunks) - len(batch_chunks)}). collection count={collection.count()}")

    return chunks[BATCH_SIZE:], ids[BATCH_SIZE:]

chunk_size = 1000
chunk_overlap = 50
BATCH_SIZE = 100
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size    = chunk_size,
    chunk_overlap = chunk_overlap,
    separators    = ['。', '\n']
)
# ドキュメント毎に、Chunkへ分割、BATCH_SIZE単位で embedding, VectorStoreへ格納
for doc_no, doc in enumerate(documents):
    fn = os.path.basename(doc.metadata['source']).split('.')[0]
    doc.metadata['source'] = fn

    # ドキュメントをチャンクへ分割
    chunks = text_splitter.split_documents([doc])
    print(f"{doc_no}: {fn}, n_chunks: {len(chunks)}")

    # Chunk毎にファイル名を付加
    for i, c in enumerate(chunks):
        chunks[i].page_content = f"# ファイル名: {fn}, Chunk No.={i+1:04}\n"+c.page_content

    # 新チャンクにIDを付けてバッファへ追加
    new_ids = [f"doc{fn}_{i+1:04}" for i in range(len(chunks))]
    pending_chunks.extend(chunks)
    pending_ids.extend(new_ids)

    # バッファ内のchunkの合計がBATCH_SIZEを超えている間、先頭からBATCH_SIZE件ずつ処理
    while len(pending_chunks) >= BATCH_SIZE:
        pending_chunks, pending_ids = embed_and_add_batch(pending_chunks, pending_ids)

# ループ終了後の残り処理
if pending_chunks:
    _ = embed_and_add_batch(pending_chunks, pending_ids)
print(f"Total collection count={collection.count()}")

3.13.12 (tags/v3.13.12:1cbe481, Feb  3 2026, 18:22:25) [MSC v.1944 64 bit (AMD64)]
google.genai=1.68.0, chromadb=1.5.5, 
langchain_community=0.4.1, langchain_text_splitters=1.1.1, 
Document files=4
0: 人事規程, n_chunks: 2
1: 出張規程, n_chunks: 2
2: 就業規則, n_chunks: 2
3: 賃金規程, n_chunks: 2
  -> 8 chunks added (total=8, remaining=0). collection count=8
Total collection count=8


### Queryの実行

In [2]:
# LLMに回答文を生成させる。
query = '始業時刻は？'
print(f"### 質問: {query}\n")

# 内部文書を検索
def retriever(query, k=3):
    # クエリをembedding
    query_emb = GEMINI_CLIENT.models.embed_content(
        model=GEMINI_EMBEDDING_MODEL,
        contents=query
    ).embeddings[0].values

    # ChromaDBで類似検索（＝retrieval）=======
    results = collection.query(
        query_embeddings=[query_emb],
        n_results = k,
        include = ['documents', 'metadatas','distances'],
    )
    print(f"\n### Retrieve results:")
    print(f"n_docs={len(results['ids'][0])}, {[i for i in results['ids'][0]]}")
    pprint(results['metadatas'][0])
    #pprint(results['documents'])
    return results

retrieved_texts = retriever(query)['documents']

# 内部文書を元にqueryを実行
prompt = f'''
後述の参照文書を参照して、質問に根拠付きで回答してください。
情報が見つからない場合には「わかりません」と回答してください。

質問: {query}

参照文書:
''' + '\n'.join(retrieved_texts[0])

response = GEMINI_CLIENT.models.generate_content(
    model=GEMINI_LLM_MODEL,
    contents=[prompt]
)
print(f"\n### 回答:\n{response.text}")

### 質問: 始業時刻は？


### Retrieve results:
n_docs=3, ['doc就業規則_0001', 'doc賃金規程_0001', 'doc人事規程_0001']
[{'source': '就業規則'}, {'source': '賃金規程'}, {'source': '人事規程'}]

### 回答:
始業時刻は午前9時です。

根拠：
就業規則 第3条第2項に「始業時刻は午前9時、終業時刻は午後6時とする。」と記載されています。
